In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from catboost import CatBoostClassifier
from xgboost import XGBClassifier


In [2]:
train = pd.read_csv(r"C:\Users\gupta\Downloads\train.csv")
test = pd.read_csv(r"C:\Users\gupta\Downloads\test.csv")

test_ids = test["PassengerId"]

df = pd.concat([train, test], axis=0).reset_index(drop=True)


In [3]:
df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)
df.drop(columns=['Cabin'], inplace=True)

df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')
df['CabinNumBin'] = pd.qcut(df['CabinNum'], 10, duplicates='drop')

In [4]:
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

# Infer CryoSleep if all spending = 0
zero_spend = (df[spend_cols].sum(axis=1) == 0)
df.loc[zero_spend & df['CryoSleep'].isna(), 'CryoSleep'] = True

# CryoSleep → spending must be 0
df.loc[df['CryoSleep'] == True, spend_cols] = 0

# Total spend
df['TotalSpend'] = df[spend_cols].sum(axis=1)

In [5]:
df['Group'] = df['PassengerId'].str.split('_').str[0]

group_size = df.groupby('Group')['PassengerId'].transform('count')
df['GroupSize'] = group_size
df['IsAlone'] = (group_size == 1).astype(int)

# Group spending behavior
df['GroupTotalSpend'] = df.groupby('Group')['TotalSpend'].transform('sum')
df['GroupAvgSpend'] = df['GroupTotalSpend'] / df['GroupSize']

In [6]:
df['Age'].fillna(df['Age'].median(), inplace=True)
df['VIP'].fillna(False, inplace=True)
df['CryoSleep'].fillna(False, inplace=True)

for col in ['HomePlanet','Destination','Deck','Side','CabinNumBin']:
    df[col].fillna(df[col].mode()[0], inplace=True)

for col in spend_cols + ['TotalSpend','GroupTotalSpend','GroupAvgSpend']:
    df[col].fillna(0, inplace=True)

C:\Users\gupta\AppData\Local\Temp\ipykernel_46492\2706387089.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\gupta\AppData\Local\Temp\ipykernel_46492\2706387089.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

In [7]:
for col in spend_cols + ['TotalSpend','GroupTotalSpend','GroupAvgSpend']:
    df[col] = np.log1p(df[col])

In [8]:
cat_cols = [
    'HomePlanet','CryoSleep','Destination',
    'VIP','Deck','Side','CabinNumBin'
]

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [9]:
df.drop(columns=['PassengerId','Name','Group'], inplace=True)

In [10]:
X = df.iloc[:len(train)].drop(columns=['Transported'])
y = train['Transported'].astype(int)
X_test = df.iloc[len(train):].drop(columns=['Transported'])

In [11]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cat_preds = np.zeros(len(X_test))
xgb_preds = np.zeros(len(X_test))

cat_scores = []
xgb_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Fold {fold+1}")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # -------- CatBoost --------
    cat = CatBoostClassifier(
        iterations=1200,
        depth=9,
        learning_rate=0.025,
        loss_function='Logloss',
        eval_metric='Accuracy',
        random_seed=42,
        verbose=0
    )

    cat.fit(X_tr, y_tr)
    val_pred = cat.predict(X_val)
    cat_scores.append(accuracy_score(y_val, val_pred))

    cat_preds += cat.predict_proba(X_test)[:,1] / kf.n_splits

    # -------- XGBoost --------
    xgb = XGBClassifier(
        n_estimators=800,
        max_depth=7,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )

    xgb.fit(X_tr, y_tr)
    val_pred = xgb.predict(X_val)
    xgb_scores.append(accuracy_score(y_val, val_pred))

    xgb_preds += xgb.predict_proba(X_test)[:,1] / kf.n_splits

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [12]:
print("CatBoost CV:", np.mean(cat_scores))
print("XGBoost CV:", np.mean(xgb_scores))

CatBoost CV: 0.8093867022765489
XGBoost CV: 0.8073150250365441


In [13]:
final_preds = (0.6 * cat_preds + 0.4 * xgb_preds) > 0.5

In [14]:
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': final_preds.astype(bool)
})

submission.to_csv("submission.csv", index=False)
print("submission.csv ready 🚀")

submission.csv ready 🚀
